# Fine-Tuning Evaluation: Base vs Fine-Tuned Agent

This notebook evaluates the effect of fine-tuning a base PPO agent using correction
data generated from a "smart agent" proxy (simulating user feedback).

We compare:
- **Base agent** (`no_doors_collect_all`)
- **Smart agent / user proxy** (`open_doors_fruits_only`)
- **Fine-tuned agent** (base agent after behavioral cloning updates)

Evaluation is done on:
1. The **same config and levels** used during correction data generation (feedback levels)
2. **Different config and levels** to test generalization

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image

# Ensure project root is importable
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import gym
import procgen  # registers envs
from stable_baselines3 import PPO
from dpu_clf import get_config_by_index, BASE_ENV_CONFIG

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

print("Imports OK")

c:\Users\matan\anaconda3\envs\procgen_env_clone\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. Load Agents

In [ ]:
# --- Agent paths (adjust finetuned path as needed) ---
BASE_AGENT_PATH = str(PROJECT_ROOT / "models/fruitbot/20260116-074523_easy/ppo_final")
SMART_AGENT_PATH = str(PROJECT_ROOT / "models/fruitbot/20251231-174002_easy/ppo_final")

# Find the latest finetuned model automatically
finetuned_root = PROJECT_ROOT / "train_env" / "finetuned_models"
if finetuned_root.exists():
    finetuned_dirs = sorted(finetuned_root.iterdir())
    if finetuned_dirs:
        latest_ft = finetuned_dirs[-1]
        # prefer best_model if it exists, else ppo_final
        if (latest_ft / "best_model.zip").exists():
            FINETUNED_AGENT_PATH = str(latest_ft / "best_model")
        else:
            FINETUNED_AGENT_PATH = str(latest_ft / "ppo_final")
        print(f"Using finetuned model: {FINETUNED_AGENT_PATH}")
    else:
        FINETUNED_AGENT_PATH = None
        print("No finetuned models found. Run finetune_from_corrections.py first.")
else:
    FINETUNED_AGENT_PATH = None
    print("No finetuned_models directory found. Run finetune_from_corrections.py first.")

# Load agents
print("\nLoading agents...")
base_agent = PPO.load(BASE_AGENT_PATH)
print(f"  Base agent loaded: no_doors_collect_all")

smart_agent = PPO.load(SMART_AGENT_PATH)
print(f"  Smart agent loaded: open_doors_fruits_only")

if FINETUNED_AGENT_PATH:
    finetuned_agent = PPO.load(FINETUNED_AGENT_PATH)
    print(f"  Finetuned agent loaded")
else:
    finetuned_agent = None
    print("  Finetuned agent NOT available")

## 2. Evaluation Helpers

In [ ]:
ACTION_NAMES = {0: "left", 1: "stay", 2: "right", 3: "throw"}


def make_env(config_index: int, seed: int, start_level: int = 0):
    """Create a procgen fruitbot environment."""
    config = get_config_by_index(config_index)
    config["rand_seed"] = seed
    config["num_levels"] = 1
    config["start_level"] = start_level
    return gym.make("procgen-fruitbot-v0", **config)


def evaluate_agent(agent, config_index, seeds, start_levels=None, name="Agent"):
    """Evaluate an agent across multiple episodes.
    
    Returns a list of dicts with per-episode metrics.
    """
    if start_levels is None:
        start_levels = [0] * len(seeds)
    
    results = []
    for i, (seed, level) in enumerate(zip(seeds, start_levels)):
        env = make_env(config_index, seed=seed, start_level=level)
        obs = env.reset()
        if isinstance(obs, tuple):
            obs = obs[0]
        
        done = False
        total_reward = 0.0
        steps = 0
        actions = []
        
        while not done:
            action, _ = agent.predict(obs, deterministic=True)
            action = int(action.item()) if hasattr(action, 'item') else int(action)
            actions.append(action)
            
            result = env.step(action)
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                done = terminated or truncated
            else:
                obs, reward, done, info = result
            
            total_reward += float(reward)
            steps += 1
        
        env.close()
        
        action_dist = Counter(actions)
        results.append({
            'agent': name,
            'seed': seed,
            'level': level,
            'config': config_index,
            'score': total_reward,
            'steps': steps,
            'n_left': action_dist.get(0, 0),
            'n_stay': action_dist.get(1, 0),
            'n_right': action_dist.get(2, 0),
            'n_throw': action_dist.get(3, 0),
        })
    
    return results


def collect_action_comparison(base, smart, finetuned, config_index, seed, start_level=0):
    """Run all three agents on the SAME env and compare their actions step-by-step."""
    agents = {'base': base, 'smart': smart}
    if finetuned is not None:
        agents['finetuned'] = finetuned
    
    all_actions = {name: [] for name in agents}
    
    # Run each agent separately on identically-seeded env
    for name, agent in agents.items():
        env = make_env(config_index, seed=seed, start_level=start_level)
        obs = env.reset()
        if isinstance(obs, tuple):
            obs = obs[0]
        done = False
        while not done:
            action, _ = agent.predict(obs, deterministic=True)
            action = int(action.item()) if hasattr(action, 'item') else int(action)
            all_actions[name].append(action)
            
            result = env.step(action)
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                done = terminated or truncated
            else:
                obs, reward, done, info = result
        env.close()
    
    return all_actions

print("Helpers defined.")

## 3. Evaluate on Feedback Levels (Config 1)

These are the same environment config and seeds used during correction data generation.

In [ ]:
# Feedback config (same as generate_corrections.py defaults)
FEEDBACK_CONFIG = 1
FEEDBACK_SEEDS = list(range(20))  # first 20 episodes
FEEDBACK_LEVELS = list(range(20))

print(f"Evaluating on FEEDBACK levels: config={FEEDBACK_CONFIG}, {len(FEEDBACK_SEEDS)} episodes")
print("="*70)

results_feedback = []
results_feedback += evaluate_agent(base_agent, FEEDBACK_CONFIG, FEEDBACK_SEEDS, FEEDBACK_LEVELS, name="Base")
results_feedback += evaluate_agent(smart_agent, FEEDBACK_CONFIG, FEEDBACK_SEEDS, FEEDBACK_LEVELS, name="Smart (User Proxy)")
if finetuned_agent:
    results_feedback += evaluate_agent(finetuned_agent, FEEDBACK_CONFIG, FEEDBACK_SEEDS, FEEDBACK_LEVELS, name="Fine-Tuned")

df_feedback = pd.DataFrame(results_feedback)

# Summary stats
summary = df_feedback.groupby('agent')['score'].agg(['mean', 'std', 'min', 'max']).round(2)
print("\nScore Summary (Feedback Levels):")
print(summary)
print()

In [ ]:
# --- Score distribution plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot of scores
agent_order = ['Base', 'Smart (User Proxy)']
if finetuned_agent:
    agent_order.append('Fine-Tuned')
colors = {'Base': '#e74c3c', 'Smart (User Proxy)': '#2ecc71', 'Fine-Tuned': '#3498db'}

sns.boxplot(data=df_feedback, x='agent', y='score', order=agent_order,
            palette=colors, ax=axes[0])
axes[0].set_title('Score Distribution – Feedback Levels (Config 1)', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Episode Score')

# Per-seed score comparison
for name in agent_order:
    subset = df_feedback[df_feedback['agent'] == name]
    axes[1].plot(subset['seed'].values, subset['score'].values,
                 'o-', label=name, color=colors[name], alpha=0.8, markersize=4)
axes[1].set_title('Per-Seed Score – Feedback Levels', fontsize=13)
axes[1].set_xlabel('Seed')
axes[1].set_ylabel('Score')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Evaluate on NEW Levels (Generalization)

Test on a different config and unseen seeds to check if fine-tuning generalizes.

In [ ]:
# Generalization test: different config and seeds
GENERAL_CONFIG = 3   # walls_doors config (very different from feedback config 1)
GENERAL_SEEDS = list(range(200, 220))  # completely unseen seeds
GENERAL_LEVELS = list(range(50, 70))   # different start levels

print(f"Evaluating on NEW levels: config={GENERAL_CONFIG}, {len(GENERAL_SEEDS)} episodes")
print("="*70)

results_general = []
results_general += evaluate_agent(base_agent, GENERAL_CONFIG, GENERAL_SEEDS, GENERAL_LEVELS, name="Base")
results_general += evaluate_agent(smart_agent, GENERAL_CONFIG, GENERAL_SEEDS, GENERAL_LEVELS, name="Smart (User Proxy)")
if finetuned_agent:
    results_general += evaluate_agent(finetuned_agent, GENERAL_CONFIG, GENERAL_SEEDS, GENERAL_LEVELS, name="Fine-Tuned")

df_general = pd.DataFrame(results_general)

summary_gen = df_general.groupby('agent')['score'].agg(['mean', 'std', 'min', 'max']).round(2)
print("\nScore Summary (Generalization Levels):")
print(summary_gen)
print()

In [ ]:
# --- Generalization score plots ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=df_general, x='agent', y='score', order=agent_order,
            palette=colors, ax=axes[0])
axes[0].set_title(f'Score Distribution – Generalization (Config {GENERAL_CONFIG})', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Episode Score')

for name in agent_order:
    subset = df_general[df_general['agent'] == name]
    axes[1].plot(subset['seed'].values, subset['score'].values,
                 'o-', label=name, color=colors[name], alpha=0.8, markersize=4)
axes[1].set_title(f'Per-Seed Score – Generalization (Config {GENERAL_CONFIG})', fontsize=13)
axes[1].set_xlabel('Seed')
axes[1].set_ylabel('Score')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Action Distribution Comparison

How does the fine-tuned agent's action profile compare to the base and smart agents?

In [ ]:
# Aggregate action distributions from feedback episodes
action_cols = ['n_left', 'n_stay', 'n_right', 'n_throw']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax_idx, (df, title_suffix) in enumerate([
    (df_feedback, f'Feedback (Config {FEEDBACK_CONFIG})'),
    (df_general, f'Generalization (Config {GENERAL_CONFIG})')
]):
    ax = axes[ax_idx]
    action_data = []
    for agent_name in agent_order:
        subset = df[df['agent'] == agent_name]
        totals = subset[action_cols].sum()
        total_actions = totals.sum()
        for col, act_name in zip(action_cols, ['Left', 'Stay', 'Right', 'Throw']):
            action_data.append({
                'Agent': agent_name,
                'Action': act_name,
                'Proportion': totals[col] / max(total_actions, 1)
            })
    
    df_actions = pd.DataFrame(action_data)
    sns.barplot(data=df_actions, x='Action', y='Proportion', hue='Agent',
                hue_order=agent_order, palette=colors, ax=ax)
    ax.set_title(f'Action Distribution – {title_suffix}', fontsize=12)
    ax.set_ylabel('Proportion')
    ax.set_ylim(0, 0.7)
    if ax_idx == 0:
        ax.legend(loc='upper right')
    else:
        ax.get_legend().remove()

plt.tight_layout()
plt.show()

## 6. Step-by-Step Action Agreement

On a single episode, compare which steps the fine-tuned agent matches the smart agent vs. the base agent.

In [ ]:
# Pick a single episode from feedback levels for detailed comparison
DEMO_SEED = 5
DEMO_LEVEL = 5

actions_dict = collect_action_comparison(
    base_agent, smart_agent, finetuned_agent,
    config_index=FEEDBACK_CONFIG, seed=DEMO_SEED, start_level=DEMO_LEVEL
)

# Compute agreement stats
min_len = min(len(v) for v in actions_dict.values())
base_arr = np.array(actions_dict['base'][:min_len])
smart_arr = np.array(actions_dict['smart'][:min_len])

print(f"Episode: config={FEEDBACK_CONFIG}, seed={DEMO_SEED}, level={DEMO_LEVEL}")
print(f"Episode length: base={len(actions_dict['base'])}, smart={len(actions_dict['smart'])}")
print(f"\nBase vs Smart agreement: {(base_arr == smart_arr).mean():.1%}")

if 'finetuned' in actions_dict:
    ft_arr = np.array(actions_dict['finetuned'][:min_len])
    print(f"Fine-Tuned vs Smart agreement: {(ft_arr == smart_arr).mean():.1%}")
    print(f"Fine-Tuned vs Base agreement:  {(ft_arr == base_arr).mean():.1%}")
    
    # Where the finetuned changed its mind compared to base
    changed = base_arr != ft_arr
    changed_to_smart = changed & (ft_arr == smart_arr)
    changed_away_from_smart = changed & (ft_arr != smart_arr)
    print(f"\nSteps where FT changed from base: {changed.sum()} / {min_len}")
    print(f"  -> Changed towards smart: {changed_to_smart.sum()}")
    print(f"  -> Changed away from smart: {changed_away_from_smart.sum()}")

In [ ]:
# Visualize action timeline for the demo episode
fig, ax = plt.subplots(figsize=(18, 4))

t = np.arange(min_len)
agent_arrays = {'Base': base_arr, 'Smart': smart_arr}
if 'finetuned' in actions_dict:
    agent_arrays['Fine-Tuned'] = ft_arr

offsets = {'Base': -0.2, 'Smart': 0.0, 'Fine-Tuned': 0.2}
plot_colors = {'Base': '#e74c3c', 'Smart': '#2ecc71', 'Fine-Tuned': '#3498db'}

for name, arr in agent_arrays.items():
    ax.scatter(t, arr + offsets.get(name, 0), label=name,
               c=plot_colors[name], alpha=0.6, s=10)

ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['Left', 'Stay', 'Right', 'Throw'])
ax.set_xlabel('Timestep')
ax.set_title(f'Action Timeline – Seed {DEMO_SEED}, Level {DEMO_LEVEL}, Config {FEEDBACK_CONFIG}')
ax.legend(loc='upper right')

# Highlight disagreement regions
if 'finetuned' in actions_dict:
    for i in range(min_len):
        if base_arr[i] != smart_arr[i]:
            ax.axvspan(i - 0.4, i + 0.4, alpha=0.08, color='orange')

plt.tight_layout()
plt.show()

## 7. Training History

Plot the fine-tuning loss and accuracy curves.

In [ ]:
# Load training history if available
history_path = None
if finetuned_root.exists():
    finetuned_dirs = sorted(finetuned_root.iterdir())
    if finetuned_dirs:
        hp = finetuned_dirs[-1] / "training_history.npz"
        if hp.exists():
            history_path = hp

if history_path:
    hist = dict(np.load(history_path))
    epochs = np.arange(1, len(hist['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Loss curves
    axes[0].plot(epochs, hist['train_loss'], 'o-', label='Train Total', color='#e74c3c')
    axes[0].plot(epochs, hist['val_loss'], 's-', label='Val Loss', color='#3498db')
    axes[0].plot(epochs, hist['train_bc_loss'], '^--', label='BC Loss', color='#e67e22', alpha=0.7)
    axes[0].plot(epochs, hist['train_kl_loss'], 'v--', label='KL Loss', color='#9b59b6', alpha=0.7)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    
    # Accuracy curves
    axes[1].plot(epochs, hist['train_acc'], 'o-', label='Train Acc', color='#e74c3c')
    axes[1].plot(epochs, hist['val_acc'], 's-', label='Val Acc', color='#3498db')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Action Prediction Accuracy (on correction data)')
    axes[1].legend()
    axes[1].set_ylim(0, 1)
    
    # BC vs KL loss ratio
    ratio = np.array(hist['train_bc_loss']) / np.maximum(np.array(hist['train_kl_loss']), 1e-8)
    axes[2].plot(epochs, ratio, 'o-', color='#1abc9c')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('BC / KL Ratio')
    axes[2].set_title('BC vs KL Loss Ratio')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nBest validation accuracy: {max(hist['val_acc']):.3f} (epoch {np.argmax(hist['val_acc'])+1})")
else:
    print("No training history found. Run finetune_from_corrections.py first.")

## 8. Summary Table

In [ ]:
# Combined summary
all_results = pd.concat([
    df_feedback.assign(eval_type='Feedback'),
    df_general.assign(eval_type='Generalization')
])

summary_table = all_results.groupby(['eval_type', 'agent']).agg(
    mean_score=('score', 'mean'),
    std_score=('score', 'std'),
    mean_steps=('steps', 'mean'),
    throw_pct=('n_throw', lambda x: x.sum() / all_results.loc[x.index, action_cols].sum().sum())
).round(3)

print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)
print(summary_table.to_string())
print("="*70)

# Highlight improvement
if finetuned_agent:
    for eval_type in ['Feedback', 'Generalization']:
        base_score = df_feedback[df_feedback['agent']=='Base']['score'].mean() if eval_type == 'Feedback' else df_general[df_general['agent']=='Base']['score'].mean()
        ft_score = df_feedback[df_feedback['agent']=='Fine-Tuned']['score'].mean() if eval_type == 'Feedback' else df_general[df_general['agent']=='Fine-Tuned']['score'].mean()
        smart_score = df_feedback[df_feedback['agent']=='Smart (User Proxy)']['score'].mean() if eval_type == 'Feedback' else df_general[df_general['agent']=='Smart (User Proxy)']['score'].mean()
        
        print(f"\n{eval_type}:")
        print(f"  Base -> Fine-Tuned: {ft_score - base_score:+.2f} score change")
        print(f"  Gap closed (towards Smart): {(ft_score - base_score) / max(smart_score - base_score, 0.01) * 100:.1f}%")